# Hardware Trojans

## Q1

### Question

In order to prevent denial of service by a hardware trojan, a test-based scheme is proposed. First, during
the testing phase, the circuit is run t times and its output is verified to be correct. If it fails, the circuit is
discareded (this is considered as a failure for the adversary). Then (usage phase), the circuit is used n > 0
times . The adversary succeeds if the circuit fails during one of the usage executions but not during the
test executions. We assume test and usage executions are indistinguishable from the point of view of the
circuit. Is this countermeasure effective ? Give a bound on the success probability of the adversary.

### Answer

The point of the adversary is to produce a malicious behavior that cannot be detected (circuit does not fail test phase) but that leaks information or does bad stuff (circuit fails usage phase).

There are two phases :
- Testing phase : run (with fake/test data) the circuit t times and check the output at each time. One failure means the circuit is discarded and the adversary has lost.
- Usage phase : use (run with real data) the circuit n > 0 times. If it fails at least once the adversary succeeds (how does failure is detected here?).
Important requirement : the circuit has no idea in which phase it is executed.

The Trojan has to behave randomly and independent from each execution so it has a probability p to fail. Thus the probability to pass the test phase is A = (1 - p)^t and the probability to pass (fail at least once) the usage phase is B = 1 - (1 - p)^n (100% minus proba of never failing). Finally, the probability to have both A and B is C = A.B = (1 - p)^t . (1 - (1 - p)^n) and this is the bound on the success probability of the adversary. If probability C is small enough then the countermeasure is effective.

In conclusion, making more repetitive tests during the test phase forces the Trojan to lower its probability to fail and then reduces the probability to fail during the usage phase. The probability success of the adversary decreases exponentially with t.

## Q2

### Question

How does your answer change if the scheme above is modified such that the number of tests t′ is drawn
uniformly at random in the set {0, . . . , t − 1} ?

### Answer

Then Pr[success|t'=rand()] = (1 - p)^t' . (1 - (1 - p)^n)

This is an improvement of the algorithm has now the adversary has to survive a random number of tests and so he cannot build attacks based on a certain number of tests as assumption.

## Q3

### Question

The scheme is improved by using 3 circuits (now called subcircuits) which receive the same input, and the
output is determined by the output given by a majority of the circuits (if there is no majority, then the
whole circuit fails). We assume that the master circuit computing the majority vote is honest. What is the
security level if the number of tests is the same for all the subcircuits ?

### Answer

Recap :
(n, k) = n! / (k!*(n - k)!)
Binomial (k success on n tests with each proba p) = (n, k) * p^k * (1 - p)^(n - k) 

If there are 3 workers where for each an adversary has a probability P = (1 - p)^t . (1 - (1 - p)^n) to pass and the adversary needs to pass at least 2 workers so the probability that he passes the master is the probability that 2 or 3 workers passes so it is P[nworkers >= 2] = P[nworkers == 2] + P[nworkers == 3] = (3, 2) * P^2 * (1 - P)^(3 - 2) + (3, 3) * P^3 * (1 - P)^(3 - 3) = 3 * (P^2 - P^3) + 1 * P^3 = 3 * P^2 - 2 * P^3

## Q4

### Question

Same question if the number of tests for each circuit is independently drawn ?

### Answer

For each worker it takes a random ti so we have for each worker :
Pi = (1 - p)^ti + (1 - (1 - p))^n

The probability for the adversary to pass is as before P[nworkers >= 2] = P[nworkers == 2] + P[nworkers == 3]

Where :

P[nworkers == 2] = P[worker0 and worker1 and not worker2] + P[worker0 and worker2 and not worker1] + P[worker1 and worker2 and not worker0] = P0 * P1 * (1 - P2) + P0 * (1 - P1) * P2 + (1 - P0) * P1 * P2 = -3 * P0 * P1 * P2 + P0 * P1 + P0 * P2 + P1 * P2

P[nworkers == 3] = P[worker0 and worker1 and worker2] = P0 * P1 * P2

Thus P[nworkers >= 2] = -2 * P0 * P1 * P2 + P0 * P1 + P0 * P2 + P1 * P2

## Secret Sharing

Secret sharing To defend against trojan activations upon inputs, we assume that the testing and usage phases
are indistinghuishable. How can that be falsified if the adversary controls the AES plaintext ?
Now, in order to prevent such an attack, the (possibly trojanized) circuits should only receive as inputs data
that is independent of the plaintext. Therefore, we use a secret sharing scheme. We split each subcircuit into 3
mini-circuits (those are the circuit coming from the malicious foundry), whose communication is managed by the
honest master. Therefore, each mini-circuit only sees at most one share, which is independent of the plaintext.
In order to perform the secret sharing (generation of shares from the plaintext v), the master unit M runs
a protocol with the mini-circuits to generate random numbers. To generate a random bit r, each mini-circuit
generates a random bit and the master computes the xor of those bits. Then, for each input bit xi, M generates
a random bit ri and sets the shares si,1 = ri and si,2 = xi ⊕ ri. The mini-circuit Γ1 and Γ2 get a share while
mini-circuit Γ0 gets nothing (it is used to compute the multiplication of shares).
The secret sharing is so performed according to ??. You are first asked understand how this secret sharing
works, answering following questions:

How can the master M operates to generate the shares of the input ?
**For each input x of n bits, the master inits the protocol and asks to all three workers to generate a random number rj | j in {0, 1, 2} of n bits and send it back to him. Once the master receives r0, r1 and r2 he computes R = r0 XOR r1 XOR r2 then for each bit i in x it creates the shares of worker 1 and worker 2 as si1 = Ri and si2 = xi XOR Ri.**

How can the master M operates to reconstruct the value hidden in a sharing ?
**The master can recover x from shares s1 and s2 very easy as x = s1 XOR s2 because each bit of x : xi = s1i XOR s2i = Ri XOR xi XOR Ri = (Ri XOR Ri) XOR xi = (0) XOR xi = xi**

In [36]:
from random import random, randint
from queue import Queue

### Logic Circuit

In [37]:
"""
LELEC2770 : Privacy Enhancing Technologies

Exercice Session : Secure 2-party computation

Logic Circuit
"""

class Gate:
    """Binary logic gate or input gate

    :param kind: "INPUT", "AND", "NAND", "OR", "NOR" or "XOR"
    :param in0_id: id of the gate connected to the first input. None for an
        input gate.
    :param in1_id: id of the gate connected to the second input. None for an
        input gate.
    """

    KINDS = ("INPUT", "AND", "NAND", "OR", "NOR", "XOR")

    def __init__(self, kind, in0_id, in1_id):
        assert kind in self.KINDS
        assert kind != "INPUT" or (in0_id is None and in1_id is None)
        self.kind = kind
        self.in0_id = in0_id
        self.in1_id = in1_id

    @classmethod
    def compute_gate(cls, kind, in1, in2):
        """Compute the output of a gate given its kind and the values at its
        input.
        """
        assert kind in cls.KINDS and kind != "INPUT"
        if kind == "AND":
            return in1 and in2
        elif kind == "NAND":
            return not (in1 and in2)
        elif kind == "OR":
            return in1 or in2
        elif kind == "NOR":
            return not (in1 or in2)
        elif kind == "XOR":
            return in1 ^ in2
        else:
            assert False


# All input gates are identical, hence this constant can be used as a shortcut.
INPUT_GATE = Gate("INPUT", None, None)


class Circuit:
    """Logic circuit

    :param g: representation of the circuit
    :param output_gates: ids of output gates
    :type g: dictionnary {gate_id, Gate}
    :type output_gates: set of ids
    """

    def __init__(self, g, output_gates):
        self.g = g
        self.output_gates = output_gates

    def evaluate(self, input_vals):
        """Evaluate logic circuit

        :param input_vals: values at the input of the circuit
        :type input_vals: dictionnary {input_gate_id: 0/1}

        :return: Circuit evalutation
        :rtype: CircuitEvaluation
        """
        for g_id, gate in self.g.items():
            assert gate.kind != "INPUT" or g_id in input_vals
        for g_id in input_vals:
            assert self.g[g_id].kind == "INPUT"
        return CircuitEvaluation(self, input_vals)


class CircuitEvaluation:
    """CircuitEvaluation

    Object created by Circuit.evaluate.
    The state attribute is a dictionnary
    {gate_id: value_at_output_of_the_gate}.
    """

    def __init__(self, circuit, input_vals):
        self.state = input_vals.copy()
        self.circuit = circuit
        for g_id in self.circuit.output_gates:
            self._recursive_evaluate(g_id)

    def _recursive_evaluate(self, g_id):
        if g_id in self.state:
            return self.state[g_id]
        else:
            in0_id = self.circuit.g[g_id].in0_id
            in1_id = self.circuit.g[g_id].in1_id
            kind = self.circuit.g[g_id].kind
            in1 = self._recursive_evaluate(in0_id)
            in2 = self._recursive_evaluate(in1_id)
            res = Gate.compute_gate(kind, in1, in2)
            self.state[g_id] = res
            return res


def test_circuit():
    """A simple test circuit"""
    circ = Circuit(
        {
            0: INPUT_GATE,
            1: INPUT_GATE,
            2: INPUT_GATE,
            3: Gate("AND", 0, 1),
            4: Gate("XOR", 2, 3),
        },
        {4},
    )
    circ_eval = circ.evaluate({0: 1, 1: 1, 2: 0})
    assert circ_eval.state == {0: 1, 1: 1, 2: 0, 3: 1, 4: 1}
    # print('Output of test circuit is', circ_eval.state[4])


if __name__ == "__main__":
    test_circuit()


### Mini-circuit

In [ ]:
"""
LELEC2770 : Privacy Enhancing Technologies

Exercice Session : Hardware Trojan

MINICIRCUIT
"""

class Minicircuit():

    def __init__(self, circuit):
        self.circuit = circuit
        self.state = {}
        self.eval_order = None

    def set_input(self, input_share):
        self.input = input_share 
        for k, v in input_share.items():
            self.state[k] = v 

    def get_random(self):
        return randint(0, 2^8) & 1

    def _recursive_evaluate(self, g_id):
        if g_id not in self.state:
            self.state[g_id] = None
            
            in0_id = self.circuit.g[g_id].in0_id
            in1_id = self.circuit.g[g_id].in1_id
            
            self._recursive_evaluate(in0_id)
            self._recursive_evaluate(in1_id)

            self.eval_order.put(g_id)


    def run_step(self):
        if self.eval_order is None:
            self.eval_order = Queue()
            for g_id in self.circuit.output_gates:
                self._recursive_evaluate(g_id)

        if self.eval_order.empty():
            return "finished"
        else:
            curr_g_id = self.eval_order.get()
            in0_id = self.circuit.g[curr_g_id].in0_id
            in1_id = self.circuit.g[curr_g_id].in1_id

            if self.circuit.g[curr_g_id].kind == "AND":
                self.curr_in = [self.state[in0_id], self.state[in1_id]]   
                self.curr_g_id = curr_g_id
                return "multiplication_protocol"
            elif self.circuit.g[curr_g_id].kind == "XOR":
                self.state[curr_g_id] = self.add(self.state[in0_id], self.state[in1_id])
                return "addition_protocol"
        
    def get_output(self):
        output = {}
        for g_id in self.circuit.output_gates:
            output[g_id] = self.state[g_id]
        return output 

class Minicircuit0(Minicircuit):

    def __init__(self, circuit):
        super().__init__(circuit)

    def gen_mul_r(self):
        r1, r2, r3, r4 = [randint(0, 1) for _ in range(4)]

        return {"r1": r1, 
                "r2": r2,
                "r3": r3,
                "r4": r4,
                "r": (r1 & r2) ^ r3 ^ r4}

class Minicircuit1(Minicircuit):
    def __init__(self, circuit):
        super().__init__(circuit)

    def add(self, in_0, in_1):
        return in_0 ^ in_1

    def mul(self, r2a, r3a, ra, da, r2b, r3b, rb, db):
        xa, xb = self.curr_in

        ea = (da & xa) ^ r3a
        eb = (db & xb) ^ r3b
        fa = xa ^ r2a
        fb = xb ^ r2b

        return ea, fa, eb, fb

class Minicircuit2(Minicircuit):
    def __init__(self, circuit):
        super().__init__(circuit)

    def add(self, in_0, in_1):
        return in_0 ^ in_1

    def mul_0(self, r1a, r1b):
        ya, yb = self.curr_in
        da = ya ^ r1a
        db = yb ^ r1b

        return da, db
    
    def mul_1(self, r1a, r4a, ea, fa, r1b, r4b, eb, fb):
        sa = r4a ^ ea ^ (fa & r1a)
        sb = r4b ^ eb ^ (fb & r1b)
        self.state[self.curr_g_id] = sa

### Master

In [ ]:
"""
LELEC2770 : Privacy Enhancing Technologies

Exercice Session : Hardware Trojan

MASTER CIRCUIT
"""

class Master():

    def __init__(self, gen_circuit, master_input):
        self.input = master_input
        self.minicircuits = [Minicircuit0(gen_circuit()), Minicircuit1(gen_circuit()), Minicircuit2(gen_circuit())]
        
    def gen_shares(self):
        input_share_0 = {}
        input_share_1 = {}
        input_share_2 = {}

        worker0: Minicircuit = self.minicircuits[0]
        worker1: Minicircuit = self.minicircuits[1]
        worker2: Minicircuit = self.minicircuits[2]

        for k, v in self.input.items():
            ri = worker0.get_random() ^ worker1.get_random() ^ worker2.get_random()
            input_share_0[k] = None # Has no shares
            input_share_1[k] = ri # si1
            input_share_2[k] = ri ^ v # si2
       
        self.minicircuits[0].set_input(input_share_0)
        self.minicircuits[1].set_input(input_share_1)
        self.minicircuits[2].set_input(input_share_2)
    
    def run(self):
        minicircuit0: Minicircuit0 = self.minicircuits[0]
        minicircuit1: Minicircuit1 = self.minicircuits[1]
        minicircuit2: Minicircuit2 = self.minicircuits[2]

        while True:
            msg1 = self.minicircuits[1].run_step()
            msg2 = self.minicircuits[2].run_step()
            
            # When would this assert break?
            assert msg1 == msg2

            if msg1 == "finished":
                break
            elif msg1 == "multiplication_protocol":
                dic = minicircuit0.gen_mul_r()
                r1, r2, r3, r4, r = dic["r1"], dic["r2"], dic["r3"], dic["r4"], dic["r"]
                r1a, r2a, r3a, r4a, ra = [randint(0, ri) for ri in (r1, r2, r3, r4, r)]
                r1b, r2b, r3b, r4b, rb = [
                    ria ^ ri for ria, ri in ((r1a, r1), (r2a, r2), (r3a, r3), (r4a, r4), (ra, r))
                ]
                da, db = minicircuit2.mul_0(r1a, r1b)
                ea, fa, eb, fb = minicircuit1.mul(r2a, r3a, ra, da, r2b, r3b, rb, db)
                minicircuit2.mul_1(r1a, r4a, ea, fa, r1b, r4b, eb, fb)
            elif msg1 == "addition_protocol":
                # XOR gates are handled locally inside run_step() via add()
                # so the master has nothing to do here.
                pass

    def reconstruct_result(self):
        output_share_0 = self.minicircuits[0].get_output()
        output_share_1 = self.minicircuits[1].get_output()
        output_share_2 = self.minicircuits[2].get_output()

        output = {}
        for k in self.minicircuits[0].circuit.output_gates:
            output[k] = output_share_1 ^ output_share_2
        return output


### Test

In [40]:
"""
LELEC2770 : Privacy Enhancing Technologies

Exercice Session : Hardware Trojan

Paper - Rock - Scissors
"""

def gen_prs_circuit():
    prs_circuit = Circuit(
        {
            "A": INPUT_GATE,
            "B": INPUT_GATE,
            "C": INPUT_GATE,
            "D": INPUT_GATE,
            "AB": Gate("AND", "A", "B"),
            "AC": Gate("AND", "A", "C"),
            "BC": Gate("AND", "B", "C"),
            "BxC": Gate("XOR", "B", "C"),
            "BD": Gate("AND", "B", "D"),
            "CD": Gate("AND", "C", "D"),
            "AD": Gate("AND", "A", "D"),
            "AxD": Gate("XOR", "A", "D"),
            "ACxBD": Gate("XOR", "AC", "BD"),
            "BCxCD": Gate("XOR", "BC", "CD"),
            "ABxAD": Gate("XOR", "AB", "AD"),
            "ACxBDxBCxCD": Gate("XOR", "ACxBD", "BCxCD"),
            "ACxBDxABxAD": Gate("XOR", "ACxBD", "ABxAD"),
            "E": Gate("XOR", "ACxBDxABxAD", "BxC"),
            "F": Gate("XOR", "ACxBDxBCxCD", "AxD"),
        },
        {"E", "F"},
    )
    return prs_circuit


def choice_to_bin(choice):
    if choice in ["PAPER", "P"]:
        return 0, 0
    elif choice in ["ROCK", "R"]:
        return 1, 0
    elif choice in ["SCISSORS", "S"]:
        return 0, 1
    elif choice in ["LOSE", "L"]:
        return 1, 1


def prs_result(E, F):
    if (E, F) == (0, 0):
        return "draw"
    elif (E, F) == (0, 1):
        return "Bob wins"
    elif (E, F) == (1, 0):
        return "Alice wins"
    else:
        raise ValueError((E, F))


def test_prs_circuit():
    inputs = {"P": (0, 0), "R": (1, 0), "S": (0, 1), "L": (1, 1)}
    for ai_n, ai in inputs.items():
        for bi_n, bi in inputs.items():
            input_all = {"A": ai[0], "B": ai[1], "C": bi[0], "D": bi[1]}
            M = Master(gen_prs_circuit, input_all)
            M.gen_shares()
            M.run()
            res = M.reconstruct_result()
            #print(ai_n, bi_n, prs_result(*res))
            ref_res = gen_prs_circuit().evaluate(input_all).state
            assert (ref_res['E'], ref_res['F']) == (res['E'], res['F'])

test_prs_circuit()

print("TESTS ARE PASSED :)")



TypeError: unsupported operand type(s) for ^: 'NoneType' and 'NoneType'